In [5]:
import os
import pandas as pd

folder_path = r"raw data\dynamic_events_pl_24"

# List all parquet files
files = [f for f in os.listdir(folder_path) if f.lower().endswith(".parquet")]
print("Parquet files found:", files)

for filename in files:
    file_path = os.path.join(folder_path, filename)
    df_name = os.path.splitext(filename)[0]  # remove .parquet extension
    try:
        globals()[df_name] = pd.read_parquet(file_path)
        print(f"DataFrame '{df_name}' created with shape {globals()[df_name].shape}")
    except Exception as e:
        print(f"Error reading {filename}: {e}")

Parquet files found: ['matches.parquet', 'players.parquet', 'teams.parquet']
DataFrame 'matches' created with shape (380, 8)
DataFrame 'players' created with shape (686, 7)
DataFrame 'teams' created with shape (20, 4)


In [9]:
# Expand 'home_team' and 'away_team' dictionaries
home_df = pd.json_normalize(matches['home_team']).add_prefix('home_')
away_df = pd.json_normalize(matches['away_team']).add_prefix('away_')

# Drop the original dict columns and concatenate the new ones
matches_expanded = pd.concat([matches.drop(['home_team', 'away_team'], axis=1), home_df, away_df], axis=1)

matches_expanded.head()

,id,date_time,status,competition_id,season_id,competition_edition_id,home_id,home_short_name,away_id,away_short_name
0,2018571,2025-05-25T15:00:00Z,closed,1,95,895,63,Bournemouth,62,Leicester
1,2018572,2025-05-25T15:00:00Z,closed,1,95,895,48,Fulham,40,Manchester City
2,2018573,2025-05-25T15:00:00Z,closed,1,95,895,752,Ipswich Town,37,West Ham
3,2018574,2025-05-25T15:00:00Z,closed,1,95,895,2,Liverpool,60,Crystal Palace
4,2018575,2025-05-25T15:00:00Z,closed,1,95,895,31,Manchester U,39,Aston Villa


In [7]:
players.head()

,id,first_name,last_name,short_name,birthday,trackable_object,gender
0,18589,Max,Aarons,M. Aarons,2000-01-04,18786,male
1,967750,Zach,Abbott,Z. Abbott,2006-05-13,969313,male
2,514681,Joshua Kofi,Acheampong,J. Acheampong,2006-05-05,516238,male
3,24629,Tyler,Adams,T. Adams,1999-02-14,25664,male
4,11589,Oluwatosin,Adarabioyo,T. Adarabioyo,1997-09-24,11670,male


In [10]:
# Expand the 'stadium' dictionary
stadium_df = pd.json_normalize(teams['stadium']).add_prefix('stadium_')

# Drop the original 'stadium' column and concatenate the new ones
teams_expanded = pd.concat([teams.drop('stadium', axis=1), stadium_df], axis=1)

teams_expanded.head()

,id,name,coach,stadium_capacity,stadium_city,stadium_id,stadium_name
0,39,Aston Villa,None,42682.0,Birmingham,488,Villa Park
1,754,Brentford FC,None,NaN,London,1970,Brentford Community Stadium
2,37,West Ham United,None,60000.0,London,52,Stade Olympique de Londres
3,31,Manchester United,None,75635.0,Manchester,63,Old Trafford
4,41,Everton,None,39571.0,Liverpool,89,Goodison Park


In [11]:
import os
import pandas as pd

# Base folder
base_folder = r"raw data"

# Folders to read
folders = ["dynamic", "meta"]

for folder in folders:
    folder_path = os.path.join(base_folder, folder)
    
    # List all Parquet files in this folder (recursive if needed)
    for root, dirs, files in os.walk(folder_path):  # os.walk allows subfolders too
        for filename in files:
            if filename.lower().endswith(".parquet"):
                file_path = os.path.join(root, filename)
                df_name = os.path.splitext(filename)[0]  # Remove .parquet
                
                # Replace any spaces or illegal characters with underscore for variable names
                df_name = df_name.replace(" ", "_").replace("-", "_")
                
                try:
                    globals()[df_name] = pd.read_parquet(file_path)
                    print(f"Loaded: {df_name} from {folder}")
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

In [24]:
import pandas as pd
import json
from pathlib import Path

meta_folder = Path("raw data/dynamic_events_pl_24/meta")
all_files = list(meta_folder.glob("*.json"))

print("Total JSON files:", len(all_files))

Total JSON files: 380


In [25]:
matches_list = []
players_list = []
periods_list = []
referees_list = []

In [26]:
for file in all_files:
    
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        match_id = data["id"]

        # -------------------
        # MATCH DATA
        # -------------------
        match_df = pd.json_normalize(
            data,
            sep="_",
            max_level=1
        )

        matches_list.append(match_df)


        # -------------------
        # PLAYERS DATA
        # -------------------
        if "players" in data:
            
            players_df = pd.json_normalize(
                data["players"],
                sep="_"
            )

            players_df["match_id"] = match_id
            players_list.append(players_df)


        # -------------------
        # PERIODS DATA
        # -------------------
        if "match_periods" in data:
            
            periods_df = pd.json_normalize(
                data["match_periods"]
            )

            periods_df["match_id"] = match_id
            periods_list.append(periods_df)


        # -------------------
        # REFEREES DATA
        # -------------------
        if "referees" in data and len(data["referees"]) > 0:
            
            refs_df = pd.json_normalize(
                data["referees"]
            )

            refs_df["match_id"] = match_id
            referees_list.append(refs_df)

    except Exception as e:
        print(f"Error reading {file.name}: {e}")

In [27]:
matches_df = pd.concat(matches_list, ignore_index=True)
players_df = pd.concat(players_list, ignore_index=True)
periods_df = pd.concat(periods_list, ignore_index=True)

if referees_list:
    referees_df = pd.concat(referees_list, ignore_index=True)

In [29]:
matches_df = matches_df.drop(
    columns=["players", "match_periods", "referees"],
    errors="ignore"
)

In [30]:
matches_df.to_parquet("matches_clean.parquet")
players_df.to_parquet("players_match.parquet")
periods_df.to_parquet("match_periods.parquet")

if 'referees_df' in globals():
    referees_df.to_parquet("referees.parquet")

In [36]:
comp_df = pd.json_normalize(matches_df['competition_edition_competition']).add_prefix('competition_')

matches_df = pd.concat(
    [matches_df.drop(columns='competition_edition_competition'), comp_df],
    axis=1
)

,id,home_team_score,away_team_score,date_time,status,home_team_side,pitch_length,pitch_width,stadium_id,stadium_name,...,competition_round_round_number,competition_round_potential_overtime,ball_trackable_object,home_team_coach,away_team_coach,competition_id,competition_area,competition_name,competition_gender,competition_age_group
0,1650385,1,0,2024-08-16T19:00:00Z,closed,"[left_to_right, right_to_left]",105,68,63,Old Trafford,...,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult
1,1650961,0,2,2024-08-17T11:30:00Z,closed,"[left_to_right, right_to_left]",106,66,481,Portman Road Stadium,...,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult
2,1651700,1,2,2024-08-17T16:30:00Z,closed,"[left_to_right, right_to_left]",105,68,52,Stade Olympique de Londres,...,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult
3,1651701,2,0,2024-08-17T14:00:00Z,closed,"[left_to_right, right_to_left]",105,68,69,Emirates,...,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult
4,1651702,0,3,2024-08-17T14:00:00Z,closed,"[right_to_left, left_to_right]",101,68,89,Goodison Park,...,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult


In [38]:
# -----------------------------
# EXPAND REMAINING DICTIONARIES
# -----------------------------

# Home kit season
home_kit_season = pd.json_normalize(matches_df["home_team_kit_season"]).add_prefix("home_kit_season_")

# Away kit season
away_kit_season = pd.json_normalize(matches_df["away_team_kit_season"]).add_prefix("away_kit_season_")

# Competition season
season_df = pd.json_normalize(matches_df["competition_edition_season"]).add_prefix("season_")

# Combine everything
matches_df = pd.concat(
    [
        matches_df.drop(columns=[
            "home_team_kit_season",
            "away_team_kit_season",
            "competition_edition_season"
        ]),
        home_kit_season,
        away_kit_season,
        season_df
    ],
    axis=1
)

# -----------------------------
# CLEAN HOME TEAM SIDE
# -----------------------------

matches_df["home_team_direction_1st_half"] = matches_df["home_team_side"].str[0]
matches_df["home_team_direction_2nd_half"] = matches_df["home_team_side"].str[1]

matches_df = matches_df.drop(columns="home_team_side")

In [39]:
dict_cols = [col for col in matches_df.columns if matches_df[col].apply(lambda x: isinstance(x, dict)).any()]
list_cols = [col for col in matches_df.columns if matches_df[col].apply(lambda x: isinstance(x, list)).any()]

print("Dictionary columns remaining:", dict_cols)
print("List columns remaining:", list_cols)

Dictionary columns remaining: []
List columns remaining: []


In [40]:
pd.set_option('display.max_columns', None)

matches_df.head()

,id,home_team_score,away_team_score,date_time,status,pitch_length,pitch_width,stadium_id,stadium_name,stadium_city,stadium_capacity,home_team_id,home_team_name,home_team_short_name,home_team_acronym,home_team_kit_id,home_team_kit_team_id,home_team_kit_name,home_team_kit_jersey_color,home_team_kit_number_color,away_team_id,away_team_name,away_team_short_name,away_team_acronym,away_team_kit_id,away_team_kit_team_id,away_team_kit_name,away_team_kit_jersey_color,away_team_kit_number_color,home_team_coach_id,home_team_coach_first_name,home_team_coach_last_name,away_team_coach_id,away_team_coach_first_name,away_team_coach_last_name,home_team_playing_time_minutes_tip,home_team_playing_time_minutes_otip,away_team_playing_time_minutes_tip,away_team_playing_time_minutes_otip,competition_edition_id,competition_edition_name,competition_round_id,competition_round_name,competition_round_round_number,competition_round_potential_overtime,ball_trackable_object,home_team_coach,away_team_coach,competition_id,competition_area,competition_name,competition_gender,competition_age_group,home_kit_season_id,home_kit_season_start_year,home_kit_season_end_year,home_kit_season_name,away_kit_season_id,away_kit_season_start_year,away_kit_season_end_year,away_kit_season_name,season_id,season_start_year,season_end_year,season_name,home_team_direction_1st_half,home_team_direction_2nd_half
0,1650385,1,0,2024-08-16T19:00:00Z,closed,105,68,63,Old Trafford,Manchester,75635,31,Manchester United,Manchester U,MUN,549,31,Home,#ff0000,#ffffff,48,Fulham,Fulham,FUL,7510,48,Home,#ffffff,#000000,326.0,Ralf,Rangnick,699.0,Scott,Parker,27.94,25.15,25.15,27.94,895,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult,5,2018,2019,2018/2019,20,2022,2022,2022,95,2024,2025,2024/2025,left_to_right,right_to_left
1,1650961,0,2,2024-08-17T11:30:00Z,closed,106,66,481,Portman Road Stadium,Ipswich,30311,752,Ipswich Town,Ipswich Town,IPS,3796,752,Home,#0000d9,#ffffff,2,Liverpool Football Club,Liverpool,LIV,250,2,home,#c10021,#ffffff,585.0,Paul,Lambert,49.0,Jürgen,Klopp,19.85,31.06,31.06,19.85,895,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult,7,2020,2021,2020/2021,4,2017,2018,2017/2018,95,2024,2025,2024/2025,left_to_right,right_to_left
2,1651700,1,2,2024-08-17T16:30:00Z,closed,105,68,52,Stade Olympique de Londres,London,60000,37,West Ham United,West Ham,WHU,142,37,home,#882739,#ffffff,39,Aston Villa,Aston Villa,AVL,3027,39,Away,#ffffff,#0a163e,872.0,David,Moyes,327.0,Steven,Gerrard,24.86,28.81,28.81,24.86,895,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult,4,2017,2018,2017/2018,7,2020,2021,2020/2021,95,2024,2025,2024/2025,left_to_right,right_to_left
3,1651701,2,0,2024-08-17T14:00:00Z,closed,105,68,69,Emirates,London,60272,3,Arsenal Football Club,Arsenal,ARS,533,3,Home alt,#ff0000,#ffffff,52,Wolverhampton Wanderers,Wolverhampton,WOL,10391,52,Away,#fbff00,#000000,869.0,Mikel,Arteta,696.0,Bruno,Lage,25.47,26.23,26.23,25.47,895,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult,5,2018,2019,2018/2019,28,2023,2024,2023/2024,95,2024,2025,2024/2025,left_to_right,right_to_left
4,1651702,0,3,2024-08-17T14:00:00Z,closed,101,68,89,Goodison Park,Liverpool,39571,41,Everton,Everton,EVE,253,41,home,#0303c8,#ffffff,308,Brighton and Hove Albion,Brighton,BHA,175,308,away,#fbe233,#000000,44.0,Rafael,Benítez,589.0,Graham,Potter,23.62,30.17,30.17,23.62,895,ENG - Premier League - 2024/2025,7,Round 1,1,False,55,NaN,NaN,1,ENG,Premier League,male,adult,4,2017,2018,2017/2018,4,2017,2018,2017/2018,95,2024,2025,2024/2025,right_to_left,left_to_right


In [23]:
players_list = []

for file in all_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        match_id = data["id"]

        players = pd.json_normalize(
            data["players"],
            sep="_"
        )

        players["match_id"] = match_id

        players_list.append(players)

    except Exception as e:
        print(f"Error reading {file.name}: {e}")

players_df = pd.concat(players_list, ignore_index=True)

print(players_df.shape)

(15176, 30)


,start_time,end_time,number,yellow_card,red_card,injured,goal,own_goal,team_player_id,team_id,...,player_role_acronym,playing_time_total_minutes_tip,playing_time_total_minutes_otip,playing_time_total_start_frame,playing_time_total_end_frame,playing_time_total_minutes_played,playing_time_total_minutes_played_regular_time,playing_time_by_period,playing_time_total,match_id
0,00:00:00,None,33,0,0,False,0,0,33982,48,...,LB,25.15,27.94,10.0,57549.0,95.88,95.88,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN,1650385
1,00:00:00,01:00:11,7,1,0,False,0,0,1062969,31,...,AM,20.42,14.54,10.0,36849.0,61.38,61.38,"[{'name': 'period_1', 'minutes_tip': 17.09, 'm...",NaN,1650385
2,00:00:00,01:03:02,32,0,0,False,0,0,1339102,48,...,CF,15.31,20.45,10.0,38559.0,64.23,64.23,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN,1650385
3,00:00:00,01:30:53,18,1,0,False,0,0,239802,48,...,RM,24.20,26.63,10.0,55269.0,92.08,92.08,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN,1650385
4,00:00:00,None,2,0,0,False,0,0,33697,48,...,RB,25.15,27.94,10.0,57549.0,95.88,95.88,"[{'name': 'period_1', 'minutes_tip': 11.42, 'm...",NaN,1650385


In [43]:
# explode periods
players_periods = players_df[['team_player_id','match_id','playing_time_by_period']].explode('playing_time_by_period')

# expand dictionary
period_details = pd.json_normalize(players_periods['playing_time_by_period'])

players_periods = pd.concat(
    [players_periods.drop(columns='playing_time_by_period').reset_index(drop=True),
     period_details.reset_index(drop=True)],
    axis=1
)

# pivot to wide format
players_periods_wide = players_periods.pivot_table(
    index=['match_id','team_player_id'],
    columns='name',
    values=['minutes_tip','minutes_otip'],
    aggfunc='first'
)

# flatten column names
players_periods_wide.columns = [
    f"{period}_{metric}" for metric, period in players_periods_wide.columns
]

players_periods_wide = players_periods_wide.reset_index()

# merge back into players dataframe
players_df = players_df.merge(
    players_periods_wide,
    on=['match_id','team_player_id'],
    how='left'
)

# optional: drop original nested column
players_df = players_df.drop(columns=['playing_time_by_period'])

In [44]:
pd.set_option('display.max_columns', None)
players_df.head()

,start_time,end_time,number,yellow_card,red_card,injured,goal,own_goal,team_player_id,team_id,id,first_name,last_name,short_name,birthday,trackable_object,gender,player_role_id,player_role_position_group,player_role_name,player_role_acronym,playing_time_total_minutes_tip,playing_time_total_minutes_otip,playing_time_total_start_frame,playing_time_total_end_frame,playing_time_total_minutes_played,playing_time_total_minutes_played_regular_time,playing_time_total,match_id,period_1_minutes_otip,period_2_minutes_otip,period_1_minutes_tip,period_2_minutes_tip
0,00:00:00,None,33,0,0,False,0,0,33982,48,17969,Antonee,Robinson,A. Robinson,1997-08-08,18166,male,19,Full Back,Left Back,LB,25.15,27.94,10.0,57549.0,95.88,95.88,NaN,1650385,17.09,10.85,11.42,13.73
1,00:00:00,01:00:11,7,1,0,False,0,0,1062969,31,13078,Mason,Mount,M. Mount,1999-01-10,13258,male,11,Midfield,Attacking Midfield,AM,20.42,14.54,10.0,36849.0,61.38,61.38,NaN,1650385,11.42,3.12,17.09,3.33
2,00:00:00,01:03:02,32,0,0,False,0,0,1339102,48,12941,Emile,Smith Rowe,E. Smith Rowe,2000-07-28,13113,male,15,Center Forward,Center Forward,CF,15.31,20.45,10.0,38559.0,64.23,64.23,NaN,1650385,17.09,3.37,11.42,3.89
3,00:00:00,01:30:53,18,1,0,False,0,0,239802,48,11438,Andreas,Hoelgebaum Pereira,Andreas Pereira,1996-01-01,11495,male,10,Midfield,Right Midfield,RM,24.20,26.63,10.0,55269.0,92.08,92.08,NaN,1650385,17.09,9.54,11.42,12.78
4,00:00:00,None,2,0,0,False,0,0,33697,48,5794,Kenny,Tete,K. Tete,1995-10-09,5804,male,20,Full Back,Right Back,RB,25.15,27.94,10.0,57549.0,95.88,95.88,NaN,1650385,17.09,10.85,11.42,13.73


In [34]:
periods_df.head()

,period,name,start_frame,end_frame,duration_frames,duration_minutes,match_id
0,1,period_1,10,27730,27720,46.20,1650385
1,2,period_2,27740,57550,29810,49.68,1650385
2,1,period_1,10,27780,27770,46.28,1650961
3,2,period_2,27790,59950,32160,53.60,1650961
4,1,period_1,10,28940,28930,48.22,1651700


In [46]:
print("Matches:", matches_df.shape)
print("Players:", players_df.shape)
print("Periods:", periods_df.shape)

if 'referees_df' in globals():
    print("Referees:", referees_df.shape)

Matches: (380, 67)
Players: (15176, 33)
Periods: (760, 7)


In [48]:
import os

dynamic_folder = r"raw data/dynamic_events_pl_24/dynamic"

parquet_files = [f for f in os.listdir(dynamic_folder) if f.endswith(".parquet")]

print("Parquet files found:")
for f in parquet_files:
    print(f)

Parquet files found:
1650385.parquet
1650961.parquet
1651700.parquet
1651701.parquet
1651702.parquet
1651703.parquet
1653146.parquet
1654741.parquet
1657085.parquet
1666058.parquet
1666917.parquet
1666918.parquet
1666919.parquet
1666920.parquet
1666921.parquet
1666922.parquet
1668692.parquet
1669604.parquet
1669605.parquet
1680834.parquet
1680836.parquet
1680837.parquet
1681563.parquet
1681564.parquet
1681565.parquet
1681566.parquet
1683027.parquet
1683028.parquet
1683029.parquet
1711918.parquet
1711920.parquet
1711921.parquet
1712846.parquet
1712847.parquet
1712849.parquet
1712850.parquet
1713834.parquet
1715919.parquet
1715920.parquet
1729513.parquet
1729514.parquet
1729515.parquet
1729516.parquet
1729517.parquet
1729518.parquet
1729520.parquet
1730528.parquet
1732598.parquet
1732599.parquet
1747407.parquet
1747409.parquet
1747410.parquet
1747411.parquet
1747412.parquet
1748494.parquet
1748495.parquet
1750708.parquet
1751820.parquet
1755069.parquet
1766002.parquet
1766988.parquet
176

In [51]:
import pandas as pd

dynamic_dfs = {}

for file in parquet_files:
    
    file_path = os.path.join(dynamic_folder, file)
    
    match_id = file.replace(".parquet", "")
    df_name = f"match_{match_id}"   # add prefix
    
    df = pd.read_parquet(file_path)
    
    dynamic_dfs[df_name] = df
    
    print(f"{df_name}: {df.shape}")

match_1650385: (4864, 294)
match_1650961: (4769, 294)
match_1651700: (4181, 294)
match_1651701: (4371, 294)
match_1651702: (4762, 294)
match_1651703: (4533, 294)
match_1653146: (4441, 294)
match_1654741: (5476, 294)
match_1657085: (5078, 294)
match_1666058: (5183, 294)
match_1666917: (4656, 294)
match_1666918: (4503, 294)
match_1666919: (5223, 294)
match_1666920: (5041, 294)
match_1666921: (4948, 294)
match_1666922: (4786, 294)
match_1668692: (4380, 294)
match_1669604: (5272, 294)
match_1669605: (4437, 294)
match_1680834: (4489, 294)
match_1680836: (5224, 294)
match_1680837: (4101, 294)
match_1681563: (5509, 294)
match_1681564: (3839, 294)
match_1681565: (4736, 294)
match_1681566: (3990, 294)
match_1683027: (5050, 294)
match_1683028: (4218, 294)
match_1683029: (4813, 294)
match_1711918: (4236, 294)
match_1711920: (4438, 294)
match_1711921: (5572, 294)
match_1712846: (4467, 294)
match_1712847: (4874, 294)
match_1712849: (4400, 294)
match_1712850: (4423, 294)
match_1713834: (5371, 294)
m

In [53]:
dynamic_dfs["match_1650385"]

,event_id,index,match_id,frame_start,frame_end,frame_physical_start,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,associated_player_possession_frame_end,associated_player_possession_end_type_id,associated_player_possession_end_type,associated_off_ball_run_event_id,associated_off_ball_run_subtype_id,associated_off_ball_run_subtype,game_state_id,game_state,team_score,opponent_team_score,phase_index,player_possession_phase_index,first_player_possession_in_team_possession,last_player_possession_in_team_possession,lead_to_different_phase,issued_from_different_phase,n_player_possessions_in_phase,team_possession_loss_in_phase,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,current_team_in_possession_next_phase_type_id,current_team_in_possession_next_phase_type,current_team_out_of_possession_next_phase_type_id,current_team_out_of_possession_next_phase_type,current_team_in_possession_previous_phase_type_id,current_team_in_possession_previous_phase_type,current_team_out_of_possession_previous_phase_type_id,current_team_out_of_possession_previous_phase_type,game_interruption_before_id,game_interruption_before,game_interruption_after_id,game_interruption_after,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,in_to_out,out_to_in,speed_avg,speed_avg_band_id,speed_avg_band,separation_start,separation_end,separation_gain,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,inside_defensive_shape_start,inside_defensive_shape_end,start_type_id,start_type,end_type_id,end_type,consecutive_on_ball_engagements,one_touch,quick_pass,carry,forward_momentum,is_header,hand_pass,initiate_give_and_go,pass_angle_received,pass_direction_received_id,pass_direction_received,pass_distance_received,pass_range_received_id,pass_range_received,pass_outcome_id,pass_outcome,targeted_passing_option_event_id,high_pass,player_targeted_id,player_targeted_name,player_targeted_position_id,player_targeted_position,player_targeted_x_pass,player_targeted_y_pass,player_targeted_channel_pass_id,player_targeted_channel_pass,player_targeted_third_pass_id,player_targeted_third_pass,player_targeted_penalty_area_pass,player_targeted_x_reception,player_targeted_y_reception,player_targeted_channel_reception_id,player_targeted_channel_reception,player_targeted_third_reception_id,player_targeted_third_reception,player_targeted_penalty_area_reception,player_targeted_distance_to_goal_start,player_targeted_distance_to_goal_end,player_targeted_angle_to_goal_start,player_targeted_angle_to_goal_end,player_targeted_average_speed,player_targeted_speed_avg_band_id,player_targeted_speed_avg_band,speed_difference,player_targeted_xpass_completion,player_targeted_difficult_pass_target,player_targeted_xthreat,player_targeted_dangerous,n_passing_options,n_off_ball_runs,n_passing_options_line_break,n_passing_options_first_line_break,n_passing_options_second_last_line_break,n_passing_options_last_line_break,n_passing_options_ahead,n_passing_options_dangerous_difficult,n_passing_options_dangerous_not_difficult,n_passing_options_not_dangerous_not_difficult,n_passing_options_not_dangerous_difficult,n_passing_options_at_start,n_passi